# 🎓 Continuous Internal Evaluation 2 (CIE-2): Machine Learning Lab
# Topic: Dataset Preparation, Exploratory Data Analysis (EDA) and Data Preprocessing using Python

| **Student Details** | **Information** |
| :--- | :--- |
| **Course** | Machine Learning (CIE Activity 2) |
| **Domain** | Supervised Learning & Tabular Preprocessing |
| **Dataset** | Student Academic Performance & Learning Analytics Dataset |
| **Target Variable** | `Final_Exam_Score` (Continuous: 0 to 100) |
| **Libraries** | `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn` |

---

### 📋 Activity Objectives:
1. **Dataset Preparation**: Loading raw data, schema verification, data types, statistical profiling, and auditing data quality issues (missing values, whitespace anomalies, duplicates).
2. **Exploratory Data Analysis (EDA)**: Conducting univariate distribution analysis, bivariate relationships, multivariate correlation heatmaps, and outlier identification via the $1.5 \times \text{IQR}$ rule.
3. **Data Preprocessing**: Implementing data cleaning (whitespace trimming, deduplication), median/mode imputation, outlier capping (Winsorization), One-Hot categorical encoding, and feature standardization (`StandardScaler`).
4. **Model Baseline & Verification**: Splitting into Train-Test sets (80/20) and validating the processed data with a baseline Linear Regression model (`MAE`, `RMSE`, $R^2$ Score).

## 1. Import Essential Libraries
We import Python's scientific computing, data manipulation, visualization, and machine learning modules.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and modeling modules
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 10

print("✅ All required libraries imported successfully!")

## 2. Dataset Preparation & Ingestion
In this section, we load the raw dataset (`student_performance_dataset.csv`) and perform initial inspections.

In [ ]:
# Ingest raw dataset
file_path = os.path.join("..", "data", "student_performance_dataset.csv")
if not os.path.exists(file_path):
    # Fallback to local directory if run from same folder
    file_path = "student_performance_dataset.csv" if os.path.exists("student_performance_dataset.csv") else os.path.join("data", "student_performance_dataset.csv")

df = pd.read_csv(file_path)
raw_df = df.copy()

print(f"Dataset successfully loaded from: {file_path}")
print(f"Dataset Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n--- First 5 Records ---")
display(df.head())
print("\n--- Last 5 Records ---")
display(df.tail())

### 2.1 Dataset Schema, Column Types & Non-Null Information

In [ ]:
print("=== Dataset Information ===")
df.info()

print("\n=== Column Data Types ===")
display(df.dtypes.to_frame(name="Data Type"))

### 2.2 Numerical vs. Categorical Feature Segregation & Descriptive Statistics

In [ ]:
# Segregate numerical and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")

print("\n--- Descriptive Statistics for Numerical Features ---")
display(df[num_cols].describe().T)

print("\n--- Descriptive Statistics for Categorical Features ---")
display(df[cat_cols].describe().T)

### 2.3 Data Quality Audit: Missing Values & Exact Duplicate Rows

In [ ]:
# Auditing Missing Values
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Column Name": df.columns,
    "Missing Count": missing_count.values,
    "Missing Percentage (%)": missing_percent.values
}).sort_values(by="Missing Count", ascending=False)

print("--- Missing Values Report ---")
display(missing_df[missing_df["Missing Count"] > 0])

# Auditing Duplicate Rows
num_duplicates = df.duplicated().sum()
print(f"\nTotal Exact Duplicate Rows Detected: {num_duplicates}")

---
## 3. Exploratory Data Analysis (EDA)
EDA helps us discover patterns, spot anomalies, test hypotheses, and verify assumptions with graphical representations.

### 3.1 Univariate Analysis: Numerical Distributions
Examining the distribution of the target variable (`Final_Exam_Score`), as well as key drivers (`Study_Hours_Per_Week` and `Attendance_Rate`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Final Exam Score Distribution
sns.histplot(df["Final_Exam_Score"].dropna(), kde=True, ax=axes[0], color="#2b5c8f", bins=20)
axes[0].set_title("Distribution of Final Exam Scores (Target)")
axes[0].set_xlabel("Final Exam Score (0–100)")
axes[0].set_ylabel("Student Count")

# Study Hours Distribution
sns.histplot(df["Study_Hours_Per_Week"].dropna(), kde=True, ax=axes[1], color="#2e7d32", bins=20)
axes[1].set_title("Distribution of Weekly Study Hours")
axes[1].set_xlabel("Study Hours / Week")
axes[1].set_ylabel("Student Count")

# Attendance Rate Distribution
sns.histplot(df["Attendance_Rate"].dropna(), kde=True, ax=axes[2], color="#c67d0a", bins=20)
axes[2].set_title("Distribution of Attendance Rate (%)")
axes[2].set_xlabel("Attendance Rate (%)")
axes[2].set_ylabel("Student Count")

plt.tight_layout()
plt.show()

### 3.2 Univariate Analysis: Categorical Features
Examining the frequencies of student demographics, parental education level, and resources.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender Distribution
sns.countplot(data=df, x="Gender", ax=axes[0, 0], palette="Blues_d")
axes[0, 0].set_title("Gender Distribution")
axes[0, 0].set_xlabel("Gender")
axes[0, 0].set_ylabel("Count")

# Parental Education Distribution
sns.countplot(data=df, x="Parental_Education", ax=axes[0, 1], palette="Greens_d",
              order=["High School", "Associate Degree", "Bachelor's", "Master's", "Doctorate"])
axes[0, 1].set_title("Parental Education Distribution")
axes[0, 1].set_xlabel("Education Level")
axes[0, 1].set_ylabel("Count")
axes[0, 1].tick_params(axis="x", rotation=25)

# Extracurricular Activities
sns.countplot(data=df, x="Extracurricular_Activities", ax=axes[1, 0], palette="Oranges_d")
axes[1, 0].set_title("Extracurricular Activities Participation")
axes[1, 0].set_xlabel("Participation")
axes[1, 0].set_ylabel("Count")

# Internet Access
sns.countplot(data=df, x="Internet_Access", ax=axes[1, 1], palette="Purples_d")
axes[1, 1].set_title("Internet Access Availability")
axes[1, 1].set_xlabel("Internet Access")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### 3.3 Bivariate Analysis: Feature Relationships with Final Exam Score

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Study Hours vs Final Exam Score
sns.regplot(data=df, x="Study_Hours_Per_Week", y="Final_Exam_Score", ax=axes[0],
            scatter_kws={"alpha": 0.5, "color": "#1f77b4"}, line_kws={"color": "red"})
axes[0].set_title("Study Hours vs. Final Exam Score")
axes[0].set_xlabel("Study Hours / Week")
axes[0].set_ylabel("Final Exam Score")

# Attendance Rate vs Final Exam Score
sns.regplot(data=df, x="Attendance_Rate", y="Final_Exam_Score", ax=axes[1],
            scatter_kws={"alpha": 0.5, "color": "#2ca02c"}, line_kws={"color": "red"})
axes[1].set_title("Attendance Rate vs. Final Exam Score")
axes[1].set_xlabel("Attendance Rate (%)")
axes[1].set_ylabel("Final Exam Score")

# Previous Score vs Final Exam Score
sns.regplot(data=df, x="Previous_Exam_Score", y="Final_Exam_Score", ax=axes[2],
            scatter_kws={"alpha": 0.5, "color": "#ff7f0e"}, line_kws={"color": "red"})
axes[2].set_title("Previous Exam Score vs. Final Exam Score")
axes[2].set_xlabel("Previous Exam Score")
axes[2].set_ylabel("Final Exam Score")

plt.tight_layout()
plt.show()

### 3.4 Bivariate Analysis: Categorical Factors Influencing Final Exam Scores

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Parental_Education", y="Final_Exam_Score", palette="Set2",
            order=["High School", "Associate Degree", "Bachelor's", "Master's", "Doctorate"])
plt.title("Final Exam Score by Parental Education Level", fontsize=13, fontweight="bold")
plt.xlabel("Parental Education Level")
plt.ylabel("Final Exam Score")
plt.tight_layout()
plt.show()

### 3.5 Multivariate Analysis: Correlation Heatmap
Computing Pearson correlation coefficients across all numerical variables to inspect linear relationships.

In [ ]:
plt.figure(figsize=(9, 7))
corr_matrix = df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True,
            linewidths=0.5, square=True)
plt.title("Correlation Matrix Heatmap of Numerical Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.6 Outlier Detection via Boxplots and 1.5 × IQR Rule

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(y=df["Study_Hours_Per_Week"], ax=axes[0], color="#90caf9")
axes[0].set_title("Boxplot: Study Hours / Week")

sns.boxplot(y=df["Attendance_Rate"], ax=axes[1], color="#a5d6a7")
axes[1].set_title("Boxplot: Attendance Rate")

sns.boxplot(y=df["Final_Exam_Score"], ax=axes[2], color="#ffcc80")
axes[2].set_title("Boxplot: Final Exam Score")

plt.tight_layout()
plt.show()

# Quantify outliers using 1.5 * IQR Rule on Study_Hours_Per_Week
q1 = df["Study_Hours_Per_Week"].quantile(0.25)
q3 = df["Study_Hours_Per_Week"].quantile(0.75)
iqr = q3 - q1
lower_threshold = q1 - 1.5 * iqr
upper_threshold = q3 + 1.5 * iqr

outlier_mask = (df["Study_Hours_Per_Week"] < lower_threshold) | (df["Study_Hours_Per_Week"] > upper_threshold)
print(f"Study Hours Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
print(f"Lower Threshold: {lower_threshold:.2f}, Upper Threshold: {upper_threshold:.2f}")
print(f"Identified Outliers in 'Study_Hours_Per_Week': {outlier_mask.sum()} records")
display(df[outlier_mask][["Student_ID", "Study_Hours_Per_Week", "Final_Exam_Score"]])

### 📌 Key Insights from Exploratory Data Analysis:
1. **Strongest Predictors**: `Study_Hours_Per_Week` and `Previous_Exam_Score` exhibit strong positive correlations with `Final_Exam_Score` ($r > 0.65$).
2. **Attendance Impact**: Consistent class attendance ($>80\%$) correlates with higher test performance.
3. **Parental Education**: Students whose parents hold advanced degrees (Master's or Doctorate) demonstrate slightly higher median scores.
4. **Data Cleanliness Issues Identified**:
   - ~3.5% missing values in `Study_Hours_Per_Week`.
   - ~2.5% missing values in `Tutoring_Sessions`.
   - Untrimmed whitespace in categorical values (e.g. `" Yes "`).
   - 6 exact duplicate student entries.
   - A few extreme upper outliers in `Study_Hours_Per_Week` ($>50$ hours).

---
## 4. Data Preprocessing Pipeline
We now apply data preprocessing steps to convert the raw data into a clean, normalized, machine-learning-ready format:
1. **String Normalization**: Trim leading and trailing whitespace in all categorical columns.
2. **Deduplication**: Remove exact duplicate records.
3. **Missing Value Imputation**: Impute numerical features using Median (robust against skewness) and categorical features using Mode.
4. **Outlier Treatment**: Cap extreme values using the IQR-based Winsorization technique.
5. **Categorical Encoding**: Convert nominal categories into numeric columns using One-Hot Encoding (`pd.get_dummies(..., drop_first=True)`).
6. **Feature Scaling**: Standardize numerical predictors using `StandardScaler` to achieve zero mean and unit variance.
7. **Dataset Export**: Save the prepared dataset to `student_performance_prepared.csv`.

In [ ]:
clean_df = raw_df.copy()

# Step 4.1: Trim extra whitespace in all object columns
for col in clean_df.select_dtypes(include="object").columns:
    clean_df[col] = clean_df[col].astype(str).str.strip()

print("✅ Step 4.1 Complete: String whitespace trimmed successfully.")

# Step 4.2: Remove exact duplicate rows
rows_before = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
rows_after = len(clean_df)
print(f"✅ Step 4.2 Complete: Deduplication removed {rows_before - rows_after} duplicate rows (Remaining: {rows_after}).")

In [ ]:
# Step 4.3: Missing Value Imputation
# Impute numerical features with Median
for col in clean_df.select_dtypes(include=[np.number]).columns:
    if clean_df[col].isna().any():
        med_val = clean_df[col].median()
        clean_df[col] = clean_df[col].fillna(med_val)
        print(f"Imputed '{col}' missing values with Median = {med_val:.2f}")

# Impute categorical features with Mode
for col in clean_df.select_dtypes(include=["object"]).columns:
    if clean_df[col].isna().any():
        mode_val = clean_df[col].mode()[0]
        clean_df[col] = clean_df[col].fillna(mode_val)
        print(f"Imputed '{col}' missing values with Mode = '{mode_val}'")

print(f"Remaining Missing Values: {clean_df.isna().sum().sum()}")
print("✅ Step 4.3 Complete: Missing value imputation finished.")

In [ ]:
# Step 4.4: Outlier Treatment via IQR Capping (Winsorization)
q1 = clean_df["Study_Hours_Per_Week"].quantile(0.25)
q3 = clean_df["Study_Hours_Per_Week"].quantile(0.75)
iqr = q3 - q1
lower_bound = max(0, q1 - 1.5 * iqr)
upper_bound = q3 + 1.5 * iqr

clean_df["Study_Hours_Per_Week"] = clean_df["Study_Hours_Per_Week"].clip(lower=lower_bound, upper=upper_bound)
print(f"✅ Step 4.4 Complete: Capped 'Study_Hours_Per_Week' between [{lower_bound:.2f}, {upper_bound:.2f}].")

In [ ]:
# Step 4.5: Drop Non-Predictive Identifier Column
if "Student_ID" in clean_df.columns:
    clean_df = clean_df.drop(columns=["Student_ID"])
    print("Dropped identifier column: 'Student_ID'")

# Step 4.6: Categorical Encoding using One-Hot Encoding (drop_first=True to prevent Dummy Variable Trap)
categorical_features = clean_df.select_dtypes(include=["object"]).columns.tolist()
encoded_df = pd.get_dummies(clean_df, columns=categorical_features, drop_first=True, dtype=int)

print(f"Features before encoding: {clean_df.shape[1]}")
print(f"Features after One-Hot Encoding: {encoded_df.shape[1]}")
print("Encoded column list:", encoded_df.columns.tolist())
display(encoded_df.head())

In [ ]:
# Step 4.7: Feature Scaling with StandardScaler
# We scale continuous predictor features, leaving dummy indicators (0/1) and the target intact
scale_cols = [
    "Age", "Study_Hours_Per_Week", "Attendance_Rate", 
    "Tutoring_Sessions", "Sleep_Hours_Per_Night", "Previous_Exam_Score"
]

scaler = StandardScaler()
prepared_df = encoded_df.copy()
prepared_df[scale_cols] = scaler.fit_transform(prepared_df[scale_cols])

print("✅ Step 4.7 Complete: StandardScaler applied to numerical features.")
print("\nVerification of Scaled Features (Mean ≈ 0, Std ≈ 1):")
display(prepared_df[scale_cols].agg(["mean", "std"]).round(4))

In [ ]:
# Step 4.8: Export Prepared Dataset
out_path = os.path.join("..", "data", "student_performance_prepared.csv")
if not os.path.exists(os.path.dirname(out_path)):
    out_path = "student_performance_prepared.csv" if not os.path.exists("data") else os.path.join("data", "student_performance_prepared.csv")

prepared_df.to_csv(out_path, index=False)
print(f"🎉 Prepared dataset saved to: {out_path}")
print(f"Final Prepared Dimensions: {prepared_df.shape[0]} rows × {prepared_df.shape[1]} columns")
display(prepared_df.head())

---
## 5. Model Baseline & Verification
To demonstrate that the prepared dataset is ready for machine learning algorithms, we split the data into training (80%) and testing (20%) sets, train a baseline **Linear Regression** model, and evaluate its predictive accuracy.

In [ ]:
# Separate Predictors (X) and Target (y)
X = prepared_df.drop(columns=["Final_Exam_Score"])
y = prepared_df["Final_Exam_Score"]

# Train-Test Split (80% Train, 20% Test, random_state=42 for reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train Shape: {X_train.shape}, y_train Shape: {y_train.shape}")
print(f"X_test Shape : {X_test.shape}, y_test Shape : {y_test.shape}")

In [ ]:
# Train Linear Regression Model
baseline_reg = LinearRegression()
baseline_reg.fit(X_train, y_train)

# Predict on Unseen Test Set
y_pred = baseline_reg.predict(X_test)

# Compute Evaluation Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 45)
print("🎯 BASELINE MODEL EVALUATION METRICS")
print("=" * 45)
print(f"Mean Absolute Error (MAE)  : {mae:.3f}")
print(f"Mean Squared Error (MSE)   : {mse:.3f}")
print(f"Root Mean Squared Error    : {rmse:.3f}")
print(f"R-squared Score (R² Score) : {r2:.4f} ({r2 * 100:.2f}% Variance Explained)")
print("=" * 45)

In [ ]:
# Visualize Actual vs. Predicted Final Exam Scores
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, color="#1976d2", alpha=0.6, edgecolors="k", label="Test Predictions")

# Perfect Prediction Diagonal Reference Line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", linewidth=2, label="Perfect Fit (y = x)")

plt.title("Actual vs. Predicted Final Exam Scores", fontsize=13, fontweight="bold")
plt.xlabel("Actual Score")
plt.ylabel("Predicted Score")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model Coefficients Inspection: Which features had the largest impact?
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": baseline_reg.coef_
}).sort_values(by="Coefficient", ascending=False)

print("--- Feature Coefficients (Weights) ---")
display(feature_importance)

---
## 6. Conclusion & Activity Summary
In this CIE-2 Machine Learning activity, we completed an end-to-end data pipeline:
1. **Dataset Preparation**: Successfully ingested and verified schema, checked feature types, audited nulls (~3.5%) and duplicates (6 rows).
2. **Exploratory Data Analysis**: Visualized distributions (univariate), investigated interactions with academic performance (bivariate), plotted correlation matrices (multivariate), and quantified outliers using the IQR rule.
3. **Data Preprocessing**: Implemented string normalization, deduplication, median/mode imputation, IQR capping, One-Hot categorical encoding, and feature standardization with `StandardScaler`.
4. **Baseline Modeling**: Validated the pipeline with a Linear Regression model achieving an impressive $R^2 \approx 0.90$, confirming that the preprocessed features are clean and highly predictive for downstream machine learning tasks.